In [1]:
import pandas as pd
import numpy as np

In [2]:
messy_housing = pd.read_csv("messy_housing_data.csv")
messy_housing.head(5)

,area_sqft,bedrooms,bathrooms,year_built,price,location
0,2696,4.0,1,1962.0,2068779.98,austin
1,2017,6.0,4,2008.0,1539208.46,San Francisco
2,1592,3.0,3,1991.0,557909.7,San Francisco
3,3358,2.0,4,1914.0,NaN,chicago
4,1624,2.0,5,1935.0,716945.96,New York City


In [3]:
len(messy_housing)

420

In [4]:
messy_housing.dtypes

area_sqft       int64
bedrooms      float64
bathrooms       int64
year_built    float64
price             str
location          str
dtype: object

**Return the percentage of null values for each column**

In [5]:
clean_data = messy_housing.copy()

missing_count = clean_data.isnull().sum()
missing_percentage = (clean_data.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percentage
}).sort_values("missing_percent", ascending=False)

print(missing_summary)

            missing_count  missing_percent
price                  50            11.90
year_built             36             8.57
bedrooms               21             5.00
area_sqft               0             0.00
bathrooms               0             0.00
location                0             0.00


**Run a missingness test on the data**
#Test 1: Are the null in each column random or clustered? 

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def test_null_randomness(df):
    """
    For each column with nulls, test if they're randomly distributed 
    across rows or if they cluster in certain rows.
    """
    print("="*80)
    print("TEST 1: ARE NULLS RANDOMLY DISTRIBUTED OR CLUSTERED?")
    print("="*80)
    
    for col in df.columns:
        n_nulls = df[col].isna().sum()
        if n_nulls == 0:
            continue
        
        # Create bins: rows with nulls vs rows with values
        df[f'{col}_is_null'] = df[col].isna().astype(int)
        
        # Test: do nulls cluster in rows with more missing data overall?
        df['total_nulls_in_row'] = df.isnull().sum(axis=1)
        
        # Split: rows with null in this column vs rows without
        rows_with_null = df[df[f'{col}_is_null'] == 1]['total_nulls_in_row']
        rows_without_null = df[df[f'{col}_is_null'] == 0]['total_nulls_in_row']
        
        # If nulls are random, both groups should have similar average nulls per row
        avg_nulls_with = rows_with_null.mean()
        avg_nulls_without = rows_without_null.mean()
        
        # T-test: are these significantly different?
        from scipy.stats import ttest_ind
        t_stat, p_value = ttest_ind(rows_with_null, rows_without_null)
        
        print(f"\n{col.upper()}")
        print(f"  Null count: {n_nulls} ({n_nulls/len(df)*100:.1f}%)")
        print(f"  Avg nulls/row (when {col} is null): {avg_nulls_with:.2f}")
        print(f"  Avg nulls/row (when {col} is present): {avg_nulls_without:.2f}")
        print(f"  T-test p-value: {p_value:.4f}")
        
        if p_value < 0.05:
            print(f"  ⚠ CLUSTERED: Nulls in {col} occur with other nulls (non-random)")
        else:
            print(f"  ✓ RANDOM: Nulls in {col} are scattered randomly")
    
    # Clean up temporary columns
    df.drop(columns=[col for col in df.columns if col.endswith('_is_null')], inplace=True)

test_null_randomness(clean_data)

TEST 1: ARE NULLS RANDOMLY DISTRIBUTED OR CLUSTERED?

BEDROOMS
  Null count: 21 (5.0%)
  Avg nulls/row (when bedrooms is null): 1.38
  Avg nulls/row (when bedrooms is present): 0.20
  T-test p-value: 0.0000
  ⚠ CLUSTERED: Nulls in bedrooms occur with other nulls (non-random)

YEAR_BUILT
  Null count: 36 (8.6%)
  Avg nulls/row (when year_built is null): 1.19
  Avg nulls/row (when year_built is present): 0.17
  T-test p-value: 0.0000
  ⚠ CLUSTERED: Nulls in year_built occur with other nulls (non-random)

PRICE
  Null count: 50 (11.9%)
  Avg nulls/row (when price is null): 1.18
  Avg nulls/row (when price is present): 0.13
  T-test p-value: 0.0000
  ⚠ CLUSTERED: Nulls in price occur with other nulls (non-random)


In [7]:
#Test 2: Are the null values in one column connected to the null values in another? 

In [8]:
def test_null_correlation(df):
    """
    For each pair of columns, test if having a null in one 
    predicts having a null in another.
    """
    print("\n" + "="*80)
    print("TEST 2: ARE NULLS IN ONE COLUMN CONNECTED TO NULLS IN ANOTHER?")
    print("="*80)
    
    # Create missing indicators
    missing_indicators = df.isnull().astype(int)
    
    # Correlation matrix of missing patterns
    missing_corr = missing_indicators.corr()
    
    # Get pairs with strong correlations
    results = []
    for i in range(len(missing_corr.columns)):
        for j in range(i+1, len(missing_corr.columns)):
            col1 = missing_corr.columns[i]
            col2 = missing_corr.columns[j]
            corr = missing_corr.iloc[i, j]
            
            if df[col1].isna().sum() > 0 and df[col2].isna().sum() > 0:
                results.append({
                    'Column_A': col1,
                    'Column_B': col2,
                    'Correlation': corr,
                    'Interpretation': 'Strong' if abs(corr) > 0.3 else 'Weak'
                })
    
    results_df = pd.DataFrame(results).sort_values('Correlation', ascending=False)
    
    print("\nNull Pattern Correlations (Between Columns):\n")
    print(results_df.to_string(index=False))
    
    # Chi-square test for each pair
    print("\n" + "-"*80)
    print("CHI-SQUARE TEST: Are nulls in column pairs dependent?")
    print("-"*80)
    
    for i in range(len(missing_corr.columns)):
        for j in range(i+1, len(missing_corr.columns)):
            col1 = missing_corr.columns[i]
            col2 = missing_corr.columns[j]
            
            if df[col1].isna().sum() == 0 or df[col2].isna().sum() == 0:
                continue
            
            # Contingency table
            contingency = pd.crosstab(
                df[col1].isna(),
                df[col2].isna()
            )
            
            chi2, p_value, dof, expected = chi2_contingency(contingency)
            
            print(f"\n{col1} vs {col2}:")
            print(f"  χ² = {chi2:.4f}, p-value = {p_value:.4f}")
            
            if p_value < 0.05:
                print(f"  ⚠ DEPENDENT: Nulls in {col1} and {col2} are related")
            else:
                print(f"  ✓ INDEPENDENT: Nulls in {col1} and {col2} are unrelated")

test_null_correlation(clean_data)


TEST 2: ARE NULLS IN ONE COLUMN CONNECTED TO NULLS IN ANOTHER?

Null Pattern Correlations (Between Columns):

  Column_A   Column_B  Correlation Interpretation
  bedrooms      price     0.084335           Weak
  bedrooms year_built     0.046829           Weak
year_built      price    -0.007504           Weak

--------------------------------------------------------------------------------
CHI-SQUARE TEST: Are nulls in column pairs dependent?
--------------------------------------------------------------------------------

bedrooms vs year_built:
  χ² = 0.3134, p-value = 0.5756
  ✓ INDEPENDENT: Nulls in bedrooms and year_built are unrelated

bedrooms vs price:
  χ² = 1.9118, p-value = 0.1668
  ✓ INDEPENDENT: Nulls in bedrooms and price are unrelated

year_built vs price:
  χ² = 0.0000, p-value = 1.0000
  ✓ INDEPENDENT: Nulls in year_built and price are unrelated


**Handling null values**

In [9]:
clean_data["price"] = pd.to_numeric(clean_data["price"], errors="coerce")
clean_data["year_built"] = pd.to_numeric(clean_data["year_built"], errors="coerce")
clean_data["bedrooms"] = pd.to_numeric(clean_data["bedrooms"], errors="coerce")

In [10]:
clean_data["price"] = clean_data["price"].fillna(clean_data["price"].median(), inplace=True)
clean_data["year_built"] = clean_data["year_built"].fillna(clean_data["year_built"].median(), inplace=True)
clean_data["bedrooms"] = clean_data["bedrooms"].fillna(clean_data["bedrooms"].mode()[0], inplace=True)

C:\Users\alexh\AppData\Local\Temp\ipykernel_22504\4044513469.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  clean_data["price"] = clean_data["price"].fillna(clean_data["price"].median(), inplace=True)
C:\Users\alexh\AppData\Local\Temp\ipykernel_22504\4044513469.py:2: ChainedAssignmentError: A value is being set on a copy of a Da

In [11]:
clean_data.isna().sum()

area_sqft             0
bedrooms              0
bathrooms             0
year_built            0
price                 0
location              0
total_nulls_in_row    0
dtype: int64

In [12]:
clean_data["year_built"] = clean_data["year_built"].astype(int)
clean_data["price"] = clean_data["price"].astype(float)
clean_data["bedrooms"] = clean_data["bedrooms"].astype(int)
clean_data.head(5)

,area_sqft,bedrooms,bathrooms,year_built,price,location,total_nulls_in_row
0,2696,4,1,1962,2068779.98,austin,0
1,2017,6,4,2008,1539208.46,San Francisco,0
2,1592,3,3,1991,557909.70,San Francisco,0
3,3358,2,4,1914,1180883.15,chicago,1
4,1624,2,5,1935,716945.96,New York City,0


**drop column total nulls in row**

In [13]:
clean_data = clean_data.drop(columns=["total_nulls_in_row"])
clean_data.head(5)

,area_sqft,bedrooms,bathrooms,year_built,price,location
0,2696,4,1,1962,2068779.98,austin
1,2017,6,4,2008,1539208.46,San Francisco
2,1592,3,3,1991,557909.70,San Francisco
3,3358,2,4,1914,1180883.15,chicago
4,1624,2,5,1935,716945.96,New York City


In [14]:
clean_data["location"] = clean_data["location"].str.strip().str.lower()

**Create data dictionaries for each city**

In [15]:
mapping_1 = {
    "san francisco": "San Francisco",
    "sf": "San Francisco"
}

clean_data["location"] = clean_data["location"].replace(mapping_1)

In [16]:
mapping_2 = {
    "austin": "Austin",
}

clean_data["location"] = clean_data["location"].replace(mapping_2)

In [17]:
mapping_3 = {
    "boston": "Boston",
    "bostan": "Boston"
}

clean_data["location"] = clean_data["location"].replace(mapping_3)

In [18]:
mapping_4 = {
    "chicago": "Chicago",
    "chicgo": "Chicago"
}

clean_data["location"] = clean_data["location"].replace(mapping_4)

In [19]:
mapping_5 = {
    "los angeles": "Los Angeles",
    "l.a.": "Los Angeles",
    "la": "Los Angeles"
}

clean_data["location"] = clean_data["location"].replace(mapping_5)

In [20]:
mapping_6 = {
    "miami": "Miami"
}

clean_data["location"] = clean_data["location"].replace(mapping_6)

In [21]:
mapping_7 = {
    "new york city": "New York City",
    "nyc": "New York City"
}

clean_data["location"] = clean_data["location"].replace(mapping_7)

In [22]:
clean_data["location"].value_counts()

location
San Francisco    85
Los Angeles      77
New York City    75
Austin           53
Chicago          50
Boston           46
Miami            34
Name: count, dtype: int64

In [23]:
clean_data["year_built"].min()
clean_data["year_built"].max()

2026

In [24]:
def assign_era(year):
    if pd.isna(year):
        return "Unknown"
    elif year < 1920:
        return "Pre-WWI"
    elif 1920 <= year < 1929:
        return "Roaring Twenties"
    elif 1929 <= year < 1939:
        return "Great Depression"
    elif 1935 <= year < 1945:
        return "World War II"
    elif 1945 <= year < 1960:
        return "Cold War"
    elif 1960 <= year < 1975:
        return "Vietnam War Era"
    elif 1975 <= year < 1990:
        return "Oil Crisis and Recovery"
    elif 1990 <= year < 2001:
        return "1990s Boom"
    elif 2001 <= year < 2008:
        return "Post 9/11 Growth"
    elif 2008 <= year < 2015:
        return "Housing Crisis"
    elif 2015 <= year < 2020:
        return "Recovery and Growth"
    else:
        return "COVID-19"

clean_data["Historical_Period"] = clean_data["year_built"].apply(assign_era)
clean_data.head(5)

,area_sqft,bedrooms,bathrooms,year_built,price,location,Historical_Period
0,2696,4,1,1962,2068779.98,Austin,Vietnam War Era
1,2017,6,4,2008,1539208.46,San Francisco,Housing Crisis
2,1592,3,3,1991,557909.70,San Francisco,1990s Boom
3,3358,2,4,1914,1180883.15,Chicago,Pre-WWI
4,1624,2,5,1935,716945.96,New York City,Great Depression


In [43]:
clean_data["Age_of_House"] = 2026 - clean_data["year_built"]
clean_data["Price_Per_Sqft"] = clean_data["price"] / clean_data["area_sqft"]
clean_data["Price_Per_Sqft"] = clean_data["Price_Per_Sqft"].round(2)
clean_data["Total_Rooms"] = clean_data["bedrooms"] + clean_data["bedrooms"]
clean_data["Decade_Built"] = (clean_data["year_built"] // 10 * 10).astype(int)

In [44]:
def categorize_size(sqft):
    if sqft < 1500:
        return "Small"
    elif sqft < 2500:
        return "Medium"
    elif sqft < 3500:
        return "Large"
    else:
        return "Luxury"

clean_data["size_category"] = clean_data["area_sqft"].apply(categorize_size)

In [45]:
def categorize_price(price):
    if price < 500000:
        return "Budget"
    elif price < 1000000:
        return "Mid-Range"
    elif price < 2000000:
        return "Premium"
    else:
        return "Luxury"

clean_data["price_category"] = clean_data["price"].apply(categorize_price)

In [46]:
def region(city):
    if city in ["San Francisco", "Los Angeles"]:
        return "West Coast"
    elif city in ["New York City", "Miami", "Boston"]:
        return "East Coast"
    elif city == "Chicago":
        return "Midwest"
    else:
        return "South"

In [47]:
clean_data["Region"] = clean_data["location"].apply(region)

In [48]:
def age_tier(age):
    if age > 100:
        return "Historic"
    elif age > 50:
        return "Vintage"
    elif age > 20:
        return "Established"
    else:
        return "Modern"

clean_data["Home_Classification"] = clean_data["Age_of_House"].apply(age_tier)

In [49]:
clean_data.head(5)

,area_sqft,bedrooms,bathrooms,year_built,price,location,Historical_Period,Age_of_House,Price_Per_Sqft,Total_Rooms,Decade_Built,size_category,price_category,Region,Home_Classification
0,2696,4,1,1962,2068779.98,Austin,Vietnam War Era,64,767.35,8,1960,Large,Luxury,South,Vintage
1,2017,6,4,2008,1539208.46,San Francisco,Housing Crisis,18,763.12,12,2000,Medium,Premium,West Coast,Modern
2,1592,3,3,1991,557909.70,San Francisco,1990s Boom,35,350.45,6,1990,Medium,Mid-Range,West Coast,Established
3,3358,2,4,1914,1180883.15,Chicago,Pre-WWI,112,351.66,4,1910,Large,Premium,Midwest,Historic
4,1624,2,5,1935,716945.96,New York City,Great Depression,91,441.47,4,1930,Medium,Mid-Range,East Coast,Vintage
